# Predicting sentiment from product reviews

# Fire up Libraries

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn import metrics

from sklearn.linear_model import LogisticRegression

# Read some product review data

## Loading reviews for a set of baby products. 

In [2]:
products = pd.read_csv('bd1.csv')

In [ ]:
products2 = pd.read_csv('bd2.csv')

In [4]:
dados = pd.read_csv('teste2.csv')

# Let's explore this data together

## Data includes the product name, the review text and the rating of the review. 

## Given that we have some lines with empty registers, we drop them

In [ ]:
products.shape

In [15]:
products.head()

,Unnamed: 0,channel0,channel1,channel2,channel3,channel4,channel5,channel6,channel7,channel8,...,ip2274,ip2275,ip2276,ip2277,ip2278,dia0,dia1,dia2,dia3,output
0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,2,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
3,3,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
4,4,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [ ]:
products.isnull().sum()

In [ ]:
products = products.dropna(axis=0) # axis = 0 means we drop lines

In [ ]:
products.isnull().sum()

In [ ]:
products.shape # we lost a few examples

In [5]:
fracao = dados[dados['is_attributed']==1].shape[0]*1.0/dados[dados['is_attributed']==0].shape[0]
classe0 = dados[dados['is_attributed']==0]
classe1 = dados[dados['is_attributed']==1]
classe0b = classe0.sample(frac=fracao, random_state=200)
classe0b.shape

(9777, 8)

In [6]:
pieces = [classe0b, classe1]
bd1 = pd.concat(pieces)
bd1 = bd1.sample(frac=1, random_state=200).reset_index(drop=True)
bd1.head()

,ip,app,device,os,channel,click_time,attributed_time,is_attributed
0,65598,3,1,8,280,2017-11-09 00:46:05,NaN,0
1,43279,14,1,10,134,2017-11-08 03:50:41,NaN,0
2,106223,35,1,19,21,2017-11-08 08:04:43,2017-11-08 08:09:59,1
3,112468,35,1,19,21,2017-11-09 10:09:13,2017-11-09 11:25:28,1
4,123733,11,1,19,173,2017-11-09 07:43:57,NaN,0


In [ ]:
fracao = bd1.shape[0]*1.0/dados.shape[0]
bd2 = dados.sample(frac=fracao, random_state=200)
bd2 = bd2.sample(frac=1, random_state=200).reset_index(drop=True)
bd2.shape

In [6]:
bd1.shape

(19554, 8)

# Examining the reviews for most-sold product:  'Vulli Sophie the Giraffe Teether'

In [ ]:
c = Counter(products['name'].values)

In [ ]:
print 'Most common:'
for name, count in c.most_common(10):
    print '%s: %7d' % (name, count)

# What this Sophie the Giraffe?

# <img src="sophie.jpg">

In [ ]:
giraffe_reviews = products[products['name'] == 'Vulli Sophie the Giraffe Teether']

In [ ]:
len(giraffe_reviews)

In [ ]:
plt.hist(giraffe_reviews['rating'], bins=np.arange(10)-0.5)
plt.xlabel("Value")
plt.xlim([-1,6])
plt.ylabel("Frequency")
plt.grid('on')
plt.show()

# Build a sentiment classifier

In [ ]:
plt.hist(products['rating'], bins=np.arange(10)-0.5)
plt.xlabel("Value")
plt.xlim([-1,6])
plt.ylabel("Frequency")
plt.grid('on')
plt.show()

## Define what's a positive and a negative sentiment

We will ignore all reviews with rating = 3, since they tend to have a neutral sentiment.  Reviews with a rating of 4 or higher will be considered positive, while the ones with rating of 2 or lower will have a negative sentiment.   

In [ ]:
#ignore all 3* reviews
products = products[products['rating'] != 3]

In [ ]:
#positive sentiment = 4* or 5* reviews
products['sentiment'] = products['rating'] >=4

In [ ]:
products.head()

In [7]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))
    
for i in range(4):
    my_features.append('dia' + str(i))

## Let's train the sentiment classifier

In [8]:
train_data = products.sample(frac=0.8, random_state=200)
test_data  = products.drop(train_data.index)
print(train_data.shape, test_data.shape)

((15643, 2998), (3911, 2998))


In [9]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

In [21]:
x1 = train_data.iloc[0:x_train.shape[0]/5,:]
x2 = train_data.iloc[x_train.shape[0]/5:2*x_train.shape[0]/5,:]
x3 = train_data.iloc[2*x_train.shape[0]/5:3*x_train.shape[0]/5,:]
x4 = train_data.iloc[3*x_train.shape[0]/5:4*x_train.shape[0]/5,:]
x5 = train_data.iloc[4*x_train.shape[0]/5:x_train.shape[0],:]

In [56]:
#xo = x1
xo = x2
xo = xo.append(x3)
xo = xo.append(x4)
xo = xo.append(x5)
xo.shape

(12515, 2998)

In [57]:
x_train = xo[my_features].values.reshape(-1,len(my_features))
y_train = xo['output'].values.reshape(-1,1)

In [ ]:
probs = classifier.predict_proba(x_train)


In [66]:
pred = [0]*probs.shape[0]
for i in range(probs.shape[0]):
    if((probs[i][1]) > 0.4):
        pred[i] = 1
    else:
        pred[i] = 0
        
print metrics.f1_score(y_train, pred)

0.906096361849


# Cross Validation

In [26]:
t = [0.3, 0.4, 0.5]
tt = [0, 0, 0]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    classifier = LogisticRegression()
    classifier.fit(x_train, y_train.ravel())

    x_test = yo[my_features].values.reshape(-1,len(my_features))
    y_test = yo['output'].values

    probs = classifier.predict_proba(x_test)

    pred = [0]*probs.shape[0]
    for i in range(probs.shape[0]):
        if((probs[i][1]) > t[z]):
            pred[i] = 1
        else:
            pred[i] = 0
    tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
    for w in range(1,5):
        xo = x1
        if(w == 1):
            yo = x2
        else:
            xo.append(x2)
        if(w == 2):
            yo = x3
        else:
            xo.append(x3)
        if(w == 3):
            yo = x4
        else:
            xo.append(x4)
        if(w == 4):
            yo = x5
        else:
            xo.append(x5)
            
        x_train = xo[my_features].values.reshape(-1,len(my_features))
        y_train = xo['output'].values.reshape(-1,1)

        classifier = LogisticRegression()
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)

        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
tt

[4.4406945069207993, 4.4796948027038814, 4.490893361420202]

# Build the word count vector for each review

In [9]:
cols = products.shape[1]
x_train = products.iloc[:,0:cols-1]
y_train = products.iloc[:,cols-1:cols]

X = np.matrix(x_train.values)
y = np.matrix(y_train.values)

In [ ]:
count_vectorizer = CountVectorizer()
x_train = count_vectorizer.fit_transform(train_data['review'].values)

# Build the y vector

In [ ]:
y_train = train_data['sentiment'].values

(15643,)

# Training the classifier

In [43]:
classifier = LogisticRegression()
classifier.fit(x_train, y_train.ravel())

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='ovr', n_jobs=1,
          penalty='l2', random_state=None, solver='liblinear', tol=0.0001,
          verbose=0, warm_start=False)

# Evaluate the sentiment model

In [45]:
x_test = test_data[my_features].values.reshape(-1,len(my_features))
y_test = test_data['output'].values

# predict class labels for the test set
predicted = classifier.predict(x_test)

In [46]:
predicted

array([1, 1, 0, ..., 1, 1, 1])

In [47]:
# generate class probabilities
probs = classifier.predict_proba(x_test)
print probs

[[ 0.00319431  0.99680569]
 [ 0.00390857  0.99609143]
 [ 0.95582212  0.04417788]
 ..., 
 [ 0.00231764  0.99768236]
 [ 0.00692359  0.99307641]
 [ 0.00684822  0.99315178]]


In [48]:
pred = [0]*probs.shape[0]
for i in range(probs.shape[0]):
    if((probs[i][1]) > 0.3):
        pred[i] = 1
    else:
        pred[i] = 0

In [50]:
# generate evaluation metrics
print metrics.accuracy_score(y_test, pred)
print metrics.roc_auc_score(y_test, probs[:, 1])
print metrics.precision_score(y_test, pred)
print metrics.recall_score(y_test, pred)
print metrics.f1_score(y_test, pred)

0.892610585528
0.9573486263
0.874572962421
0.916624040921
0.895104895105


# Applying the learned model to understand sentiment for Giraffe

In [ ]:
sophie_probs = classifier.predict_proba(count_vectorizer.transform(giraffe_reviews['review']))

In [ ]:
sophie_probs.shape

In [ ]:
giraffe_reviews['predicted_sentiment'] = sophie_probs.max(axis=1)

In [ ]:
giraffe_reviews.head()

## Sort the reviews based on the predicted sentiment and explore

In [ ]:
giraffe_reviews = giraffe_reviews.sort_values('predicted_sentiment', ascending=False)

## Most positive reviews for the giraffe

In [ ]:
giraffe_reviews.head()

In [ ]:
print giraffe_reviews[giraffe_reviews.index==34892]['review'].values

In [ ]:
print giraffe_reviews[giraffe_reviews.index==34434]['review'].values

## Show most negative reviews for giraffe

In [ ]:
giraffe_reviews.tail()

In [ ]:
print giraffe_reviews[giraffe_reviews.index==34576]['review'].values

In [ ]:
print giraffe_reviews[giraffe_reviews.index==35072]['review'].values

# Quizz questions

### Answer 1

### Answer 2

### Answer 3

### Answer 4